# Línea Base para Predicción de Combates Pokémon

### Importación de Módulos

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import csr_matrix

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

### Carga de Datos

In [13]:
df_cleaned = pd.read_csv("../data/gen9ou_cleaned_dataset.csv")
df_cleaned.head()

,battle_id,p1_poke1,p1_poke2,p1_poke3,p1_poke4,p1_poke5,p1_poke6,p2_poke1,p2_poke2,p2_poke3,p2_poke4,p2_poke5,p2_poke6,p1_win
0,1636793-gen9ou-2460221181,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,1
1,1636793-gen9ou-2460221181,Porygon-Z,Great Tusk,Raging Bolt,Corviknight,Meowscarada,Dragonite,Dragapult,Kingambit,Hawlucha,Glimmora,Chesnaught,Gallade,0
2,95801-gen9ou-2513923983,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,1
3,95801-gen9ou-2513923983,Hawlucha,Rillaboom,Azumarill,Iron Moth,Corviknight,Raging Bolt,Meganium,Arboliva,Avalugg,Ditto,Houndoom,Tinkaton,0
4,1514712-gen9ou-2429668049,Dragonite,Zamazenta,LandorusTherianTherian,Gholdengo,Darkrai,Hatterene,Ninetales,Ceruledge,Venusaur,Hatterene,Great Tusk,Walking Wake,1


## 1. Separación de Conjuntos de Entrenamiento y Prueba

Split según battle_id para evitar data leakage

In [14]:
battle_ids = df_cleaned['battle_id'].unique()

train_val_ids, test_ids = train_test_split(
    battle_ids,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_ids, val_ids = train_test_split(
    train_val_ids,
    test_size=0.2, 
    random_state=RANDOM_STATE
)

train_df = df_cleaned[df_cleaned["battle_id"].isin(train_ids)]
val_df   = df_cleaned[df_cleaned["battle_id"].isin(val_ids)]
test_df  = df_cleaned[df_cleaned["battle_id"].isin(test_ids)]

## 2. Encoding

In [15]:
pokemon_cols = [
    "p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6",
    "p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"
]

all_pokemon = pd.unique(df_cleaned[pokemon_cols].values.ravel())

pokemon_to_idx = {p: i for i, p in enumerate(sorted(all_pokemon))}
n_pokemon = len(pokemon_to_idx)

print("Número de Pokémon:", n_pokemon)

Número de Pokémon: 809


Se codifica el dataset utilizando Multi-hot encoding (manual) con una representación de 1 para cada Pokémon presente en el combate y 0 para los ausentes. Esto se hace para ambos equipos (equipo 1 y equipo 2) y se concatenan las representaciones para formar la matriz de características final.

Además, se utiliza `scipy.sparse.csr_matrix` para convertir la matriz de características a un formato disperso, lo que es eficiente en términos de memoria dado que la mayoría de los valores serán ceros.

In [16]:
def build_sparse_matrix(df, pokemon_to_idx):
    n_rows = len(df)
    n_poke_unique = len(pokemon_to_idx)

    p1_cols = ["p1_poke1","p1_poke2","p1_poke3","p1_poke4","p1_poke5","p1_poke6"]
    p2_cols = ["p2_poke1","p2_poke2","p2_poke3","p2_poke4","p2_poke5","p2_poke6"]

    p1_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p1_cols], axis=1)
    p2_idx = np.stack([df[c].map(pokemon_to_idx).values for c in p2_cols], axis=1)

    # Offset para el jugador 2
    p2_idx = p2_idx + n_poke_unique

    # Cada fila tendrá 12 índices (6 de p1 y 6 de p2)
    all_indices = np.hstack([p1_idx, p2_idx]) 

    # 'rows' debe repetirse 12 veces por cada fila del DF
    rows = np.repeat(np.arange(n_rows), 12)
    cols = all_indices.flatten()
    data = np.ones(len(rows), dtype=np.int8)

    X = csr_matrix(
        (data, (rows, cols)),
        shape=(n_rows, 2 * n_poke_unique),
        dtype=np.int8
    )

    y = df["p1_win"].to_numpy()
    return X, y

In [17]:
X_train, y_train = build_sparse_matrix(train_df, pokemon_to_idx)
X_val, y_val     = build_sparse_matrix(val_df, pokemon_to_idx)
X_test, y_test   = build_sparse_matrix(test_df, pokemon_to_idx)

print(X_train.shape)

(2255146, 1618)


A continuación, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline de Random Forest, debido a limitaciones de memoria y tiempo de entrenamiento.

In [18]:
RF_SAMPLE_BATTLES = 100_000

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

In [19]:
X_rf_train, y_rf_train = build_sparse_matrix(rf_df, pokemon_to_idx)

## 3. Entrenamiento del Modelo de Baseline

### 3.2 Entrenamiento de Random Forest

Ya que Random Forest no puede manejar matrices dispersas, convertimos a formato denso (aunque esto puede consumir mucha memoria, por lo que se recomienda hacerlo solo con una muestra de los datos).

In [20]:
X_rf_train_dense = X_rf_train.toarray()
X_val_dense = X_val.toarray()
X_test_dense = X_test.toarray()

Se define a continuación el la función objetivo para Optuna, que busca los mejores hiperparámetros para el modelo de Random Forest utilizando la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [21]:
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

def rf_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_rf_train, y_rf_train)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [ ]:
study_rf = optuna.create_study(
    study_name="rf_roc_auc",
    direction="maximize",
    storage="sqlite:///optuna_rf_classic.db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective, n_trials=5)


[I 2026-03-23 13:02:48,066] A new study created in RDB with name: rf_roc_auc
[I 2026-03-23 13:03:15,972] Trial 0 finished with value: 0.5465956581884756 and parameters: {'n_estimators': 491, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5465956581884756.
[I 2026-03-23 13:04:38,476] Trial 1 finished with value: 0.553169612270336 and parameters: {'n_estimators': 546, 'max_depth': 22, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.553169612270336.
[I 2026-03-23 13:04:47,247] Trial 2 finished with value: 0.5430364375512442 and parameters: {'n_estimators': 230, 'max_depth': 5, 'min_samples_split': 5, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.553169612270336.
[I 2026-03-23 13:05:15,918] Trial 3 finished with value: 0.5521422729525126 and parameters: {'n_estimators': 180, 'max_depth': 22, 'min_samples_split': 10, 'min_samples_leaf'

In [24]:
best_params = study_rf.best_params.copy()

rf_model = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model.fit(X_rf_train_dense, y_rf_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",546
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",22
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",8
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",5
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

Tras el entrenamiento con los mejores hiperparámetros, se evalúa el modelo en el conjunto de prueba.

In [25]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    log_loss,
    confusion_matrix
)

val_proba = rf_model.predict_proba(X_val_dense)[:, 1]


thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [26]:


test_proba = rf_model.predict_proba(X_test_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5371
F1 macro:  0.5356
ROC-AUC:   0.5541
Log Loss:  0.6906

Matriz de confusión:
[[169318 183049]
 [143199 209168]]


Para comparar métricas, se incluye dos baseline naive que predice siempre la clase mayoritaria del conjunto de entrenamiento, y otro que predice aleatoriamente según la distribución de clases del conjunto de entrenamiento.

In [28]:
majority_class = np.bincount(y_train).argmax()
y_pred_naive = np.full_like(y_test, majority_class)
y_pred_random = np.random.randint(0, 2, size=len(y_test))


print("Naive Accuracy:", accuracy_score(y_test, y_pred_naive))
print("Naive F1 macro:", f1_score(y_test, y_pred_naive, average="macro"))
print("Random Accuracy:", accuracy_score(y_test, y_pred_random))
print("Random F1 macro:", f1_score(y_test, y_pred_random, average="macro"))

Naive Accuracy: 0.5
Naive F1 macro: 0.3333333333333333
Random Accuracy: 0.5005690090161679
Random F1 macro: 0.5005686683952584


### 3.3 Entrenamiento de LightGBM

Dado que LightGBM puede manejar matrices dispersas, se entrena directamente con la matriz en formato CSR sin necesidad de convertir a formato denso, lo que es más eficiente en términos de memoria. Por lo tanto, se puede entrenar con un mayor número de muestras sin preocuparse por el consumo de memoria.

In [32]:
# Convertir los conjuntos de entrenamiento, validación y prueba
X_train_float = X_train.astype('float32')
X_val_float = X_val.astype('float32')
X_test_float = X_test.astype('float32')

# Nota: y_train, y_val, y_test suelen estar bien como int o float, 
# pero si quieres asegurar compatibilidad total:
y_train_float = y_train.astype('float32')
y_val_float = y_val.astype('float32')
y_test_float = y_test.astype('float32')

Por compatibilidad con LightGBM, se convierte el tipo de datos de las características a `float32`.

In [33]:
import lightgbm as lgb

def lgb_objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_float,
        y_train_float,
        eval_set=[(X_val_float, y_val_float)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_float)[:,1]

    roc_auc = roc_auc_score(y_val_float, val_proba)

    return roc_auc

Definimos la función objetivo de Optuna para optimizar los hiperparámetros de LightGBM, además de escoger el mejor threshold para convertir las probabilidades de salida en predicciones binarias. Se utiliza la métrica F1-score para evaluar el rendimiento del modelo durante la optimización.

In [34]:
study = optuna.create_study(
    study_name="lgb_roc_auc",
    direction="maximize",
    storage="sqlite:///optuna_lgb_classic.db",
    load_if_exists=True
)

study.optimize(lgb_objective, n_trials=10)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-23 14:07:27,865] Using an existing study with name 'lgb_roc_auc' instead of creating a new one.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.414037 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1949
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 974
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[311]	valid_0's auc: 0.577461


[I 2026-03-23 14:08:38,001] Trial 2 finished with value: 0.5774614415706718 and parameters: {'n_estimators': 311, 'learning_rate': 0.04044518675968803, 'num_leaves': 93, 'max_depth': 26, 'min_child_samples': 197, 'subsample': 0.8310165233148682, 'colsample_bytree': 0.7675039527994292}. Best is trial 2 with value: 0.5774614415706718.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.444418 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2103
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1051
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[1073]	valid_0's auc: 0.588296


[I 2026-03-23 14:11:02,862] Trial 3 finished with value: 0.588295739926624 and parameters: {'n_estimators': 1073, 'learning_rate': 0.037925408981351515, 'num_leaves': 103, 'max_depth': 26, 'min_child_samples': 124, 'subsample': 0.7775623699854571, 'colsample_bytree': 0.928082192884104}. Best is trial 3 with value: 0.588295739926624.


...

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 2.214983 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2909
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1454
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[233]	valid_0's auc: 0.588316


[I 2026-03-23 14:19:47,805] Trial 10 finished with value: 0.5883159842812972 and parameters: {'n_estimators': 233, 'learning_rate': 0.13079709551701937, 'num_leaves': 241, 'max_depth': 26, 'min_child_samples': 19, 'subsample': 0.7018813348151991, 'colsample_bytree': 0.9214319355737466}. Best is trial 5 with value: 0.6009225813985741.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.668883 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2051
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1025
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[746]	valid_0's auc: 0.583992


[I 2026-03-23 14:20:57,414] Trial 11 finished with value: 0.5839920333293858 and parameters: {'n_estimators': 747, 'learning_rate': 0.14600758378624293, 'num_leaves': 33, 'max_depth': 0, 'min_child_samples': 137, 'subsample': 0.6104908925109552, 'colsample_bytree': 0.8361490326913242}. Best is trial 5 with value: 0.6009225813985741.


Best params: {'n_estimators': 1001, 'learning_rate': 0.14568135522223793, 'num_leaves': 232, 'max_depth': 26, 'min_child_samples': 160, 'subsample': 0.8687590985345857, 'colsample_bytree': 0.6702925308347302}
Best score: 0.6009225813985741


Una vez obtenidos los mejores hiperparámetros, se entrena el modelo final de LightGBM con esos parámetros y se evalúa su rendimiento en el conjunto de test.

In [35]:
best_model = lgb.LGBMClassifier(**study.best_params)

best_model.fit(
    X_train_float,
    y_train_float,
    eval_set=[(X_val_float, y_val_float)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model.predict_proba(X_val_float)[:, 1]

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.521684 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2014
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 1007
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits wit

Una vez entrenado el modelo, se ajusta el threshold en función de F1-score macro.

In [36]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.502


In [38]:
test_proba = best_model.predict_proba(X_test_float)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test_float, test_preds)
f1_macro = f1_score(y_test_float, test_preds, average="macro")
roc_auc = roc_auc_score(y_test_float, test_proba)
ll = log_loss(y_test_float, test_proba)
cm = confusion_matrix(y_test_float, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5702
F1 macro:  0.5702
ROC-AUC:   0.6002
Log Loss:  0.6768

Matriz de confusión:
[[203804 148563]
 [154313 198054]]


## 4. Representación Team Differential

Se basa en la idea de representar cada combate como la diferencia entre los pokémon del equipo 1 y los pokémon del equipo 2. Esto se hace restando la representación de los pokémon del equipo 2 de la representación de los pokémon del equipo 1 de la forma `team_diff_vector = team1_vector - team2_vector`, lo que da como resultado una nueva representación que captura las diferencias entre ambos equipos. 

In [39]:
def build_team_differential(df, pokemon_to_idx):
    n_pokemon = len(pokemon_to_idx)
    
    rows = []
    cols = []
    data = []
    
    y = df['p1_win'].values
    
    for row_idx, row in enumerate(df.itertuples(index=False)):
        # team 1: +1
        for p in [row.p1_poke1,row.p1_poke2,row.p1_poke3,row.p1_poke4,row.p1_poke5,row.p1_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(1)
        
        # team 2: -1
        for p in [row.p2_poke1,row.p2_poke2,row.p2_poke3,row.p2_poke4,row.p2_poke5,row.p2_poke6]:
            cols.append(pokemon_to_idx[p])
            rows.append(row_idx)
            data.append(-1)
    
    X = csr_matrix((data, (rows, cols)), shape=(len(df), n_pokemon), dtype=np.int8)
    
    return X, y

Se introduce el experimento de eliminar la duplicación de datos, ya que con esta representación no es necesario invertir los equipos para evitar el sesgo del orden, dado que la diferencia ya captura esa información.

In [45]:
# =========================
# ELIMINAR DUPLICADOS POR BATTLE_ID
# =========================

def remove_battle_duplicates(df):
    print(f"Original size: {len(df)}")
    
    df_unique = df.drop_duplicates(subset="battle_id", keep="first").copy()
    
    print(f"After removing duplicates: {len(df_unique)}")
    print(f"Removed: {len(df) - len(df_unique)} rows")
    
    return df_unique


# Aplicar a cada split
train_df_nodup = remove_battle_duplicates(train_df)
val_df_nodup   = remove_battle_duplicates(val_df)
test_df_nodup  = remove_battle_duplicates(test_df)

Original size: 2255146
After removing duplicates: 1127573
Removed: 1127573 rows
Original size: 563788
After removing duplicates: 281894
Removed: 281894 rows
Original size: 704734
After removing duplicates: 352367
Removed: 352367 rows


In [46]:
# Train completo
X_train_diff, y_train = build_team_differential(train_df, pokemon_to_idx)
X_test_diff, y_test = build_team_differential(test_df, pokemon_to_idx)
X_val_diff, y_val = build_team_differential(val_df, pokemon_to_idx)

#hay que meter el conjunto de validación también
#INTERESANTE PROBAR SI CON ESTA REPRESENTACION ES NECESARIA LA DUPLICACION

print("Team Differential shapes:")
print("X_train:", X_train_diff.shape)
print("X_test :", X_test_diff.shape)
print("X_val  :", X_val_diff.shape)

Team Differential shapes:
X_train: (2255146, 809)
X_test : (704734, 809)
X_val  : (563788, 809)


Al igual que antes, se selecciona una muestra de 100000 combates para entrenar el modelo de baseline con esta nueva representación.

In [47]:
RF_SAMPLE_BATTLES = 100_000  # ajustar según memoria

rf_battle_ids = rng.choice(
    train_df["battle_id"].unique(),
    size=RF_SAMPLE_BATTLES,
    replace=False
)

rf_df = train_df[train_df["battle_id"].isin(rf_battle_ids)]

X_rf_diff, y_rf = build_team_differential(rf_df, pokemon_to_idx)

# Random Forest necesita matriz densa
X_train_rf_diff_dense = X_rf_diff.toarray()
X_test_rf_diff_dense = X_test_diff.toarray()
X_val_rf_diff_dense = X_val_diff.toarray()

## 5. Entrenamiento de Modelos con Representación Team Differential

### 5.1 Entrenamiento de Random Forest con Team Differential

In [48]:
def rf_objective_diff(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 25), 
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "class_weight": "balanced",
        "n_jobs": -1,
        "random_state": RANDOM_STATE
    }
    
    model = RandomForestClassifier(**params)
    model.fit(X_train_rf_diff_dense, y_rf_train)

    # Probabilidades (clave para ROC-AUC)
    val_proba = model.predict_proba(X_val_rf_diff_dense)[:, 1]

    # Métrica independiente de threshold
    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [49]:
study_rf = optuna.create_study(
    study_name="rf_roc_auc",
    direction="maximize",
    storage="sqlite:///optuna_rf_diff.db",
    load_if_exists=True,
)

study_rf.optimize(rf_objective_diff, n_trials=5)

[I 2026-03-23 15:49:42,457] A new study created in RDB with name: rf_roc_auc
[I 2026-03-23 15:51:11,401] Trial 0 finished with value: 0.5508220478317368 and parameters: {'n_estimators': 167, 'max_depth': 21, 'min_samples_split': 7, 'min_samples_leaf': 2, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5508220478317368.
[I 2026-03-23 15:52:15,899] Trial 1 finished with value: 0.5472180033332049 and parameters: {'n_estimators': 189, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5508220478317368.
[I 2026-03-23 15:54:11,175] Trial 2 finished with value: 0.5456181479527149 and parameters: {'n_estimators': 531, 'max_depth': 8, 'min_samples_split': 4, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5508220478317368.
[I 2026-03-23 15:55:06,921] Trial 3 finished with value: 0.5480448039928889 and parameters: {'n_estimators': 456, 'max_depth': 11, 'min_samples_split': 5, 'min_samples_lea

In [54]:
best_params = study_rf.best_params.copy()

rf_model_diff = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

rf_model_diff.fit(X_train_rf_diff_dense, y_rf_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",552
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",4
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [55]:
val_proba = rf_model_diff.predict_proba(X_val_rf_diff_dense)[:, 1]

thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [56]:
test_proba = rf_model_diff.predict_proba(X_test_rf_diff_dense)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test, test_preds)
f1_macro = f1_score(y_test, test_preds, average="macro")
roc_auc = roc_auc_score(y_test, test_proba)
ll = log_loss(y_test, test_proba)
cm = confusion_matrix(y_test, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5340
F1 macro:  0.5273
ROC-AUC:   0.5519
Log Loss:  0.6918

Matriz de confusión:
[[146110 206257]
 [122136 230231]]


### 5.2 Entrenamiento de LightGBM con Team Differential

In [57]:
X_train_diff = X_train_diff.astype('float32')
X_test_diff = X_test_diff.astype('float32')
X_val_diff = X_val_diff.astype('float32')

In [58]:
import lightgbm as lgb

def lgb_objective_diff(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", -1, 30),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "n_jobs": -1,
        "random_state": RANDOM_STATE,
        "objective": "binary",
        "metric": "auc"
    }


    model = lgb.LGBMClassifier(**params)

    model.fit(
        X_train_diff,
        y_train,
        eval_set=[(X_val_diff, y_val)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(50),
            lgb.log_evaluation(0)
        ]
    )

    val_proba = model.predict_proba(X_val_diff)[:,1]

    roc_auc = roc_auc_score(y_val, val_proba)

    return roc_auc

In [59]:
study = optuna.create_study(
    study_name="lgb_roc_auc",
    direction="maximize",
    storage="sqlite:///optuna_lgb_diff.db",
    load_if_exists=True
)

study.optimize(lgb_objective_diff, n_trials=10)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-23 21:44:44,643] A new study created in RDB with name: lgb_roc_auc


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.822363 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1842
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 614
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[674]	valid_0's auc: 0.588835


[I 2026-03-23 21:46:41,564] Trial 0 finished with value: 0.5888347215349494 and parameters: {'n_estimators': 675, 'learning_rate': 0.07200362424604095, 'num_leaves': 179, 'max_depth': -1, 'min_child_samples': 66, 'subsample': 0.9983260197225596, 'colsample_bytree': 0.671907503770881}. Best is trial 0 with value: 0.5888347215349494.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.891199 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1552
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 517
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[321]	valid_0's auc: 0.568515


[I 2026-03-23 21:47:36,998] Trial 1 finished with value: 0.568514760337762 and parameters: {'n_estimators': 321, 'learning_rate': 0.03966433448694395, 'num_leaves': 47, 'max_depth': 27, 'min_child_samples': 150, 'subsample': 0.8357430764210766, 'colsample_bytree': 0.785215773848373}. Best is trial 0 with value: 0.5888347215349494.


...

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.899251 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1537
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

[I 2026-03-23 22:00:45,163] Trial 8 finished with value: 0.5857216360925488 and parameters: {'n_estimators': 961, 'learning_rate': 0.04645473903158645, 'num_leaves': 115, 'max_depth': 20, 'min_child_samples': 161, 'subsample': 0.8767202818914195, 'colsample_bytree': 0.6093083263012058}. Best is trial 3 with value: 0.5927139432643006.


[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.985129 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1744
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 581
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

[I 2026-03-23 22:02:18,179] Trial 9 finished with value: 0.5907056338142378 and parameters: {'n_estimators': 537, 'learning_rate': 0.14893687804968253, 'num_leaves': 174, 'max_depth': 23, 'min_child_samples': 86, 'subsample': 0.6199070323963671, 'colsample_bytree': 0.665187226202598}. Best is trial 3 with value: 0.5927139432643006.


Best params: {'n_estimators': 964, 'learning_rate': 0.12310988986788557, 'num_leaves': 155, 'max_depth': 30, 'min_child_samples': 53, 'subsample': 0.7077603108821007, 'colsample_bytree': 0.6574352620620908}
Best score: 0.5927139432643006


In [60]:
best_model_lgbm = lgb.LGBMClassifier(**study.best_params)

best_model_lgbm.fit(
    X_train_diff,
    y_train,
    eval_set=[(X_val_diff, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)]
)

val_proba = best_model_lgbm.predict_proba(X_val_diff)[:, 1]

[LightGBM] [Info] Number of positive: 1127573, number of negative: 1127573
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.379628 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1905
[LightGBM] [Info] Number of data points in the train set: 2255146, number of used features: 636
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Did not meet early stopping. Best iteration is:
[960]	valid_0's auc: 0.592732	valid_0's binary_logloss: 0.678714


In [61]:
thresholds = np.linspace(0.3, 0.7, 100)

best_thr = 0.5
best_f1_macro = 0

for thr in thresholds:
    preds = (val_proba >= thr).astype(int)
    f1_macro = f1_score(y_val, preds, average="macro")
    
    if f1_macro > best_f1_macro:
        best_f1_macro = f1_macro
        best_thr = thr

print(f"Best threshold (F1 macro): {best_thr:.3f}")

Best threshold (F1 macro): 0.498


In [62]:
test_proba = best_model.predict_proba(X_test_float)[:, 1]

test_preds = (test_proba >= best_thr).astype(int)

acc = accuracy_score(y_test_float, test_preds)
f1_macro = f1_score(y_test_float, test_preds, average="macro")
roc_auc = roc_auc_score(y_test_float, test_proba)
ll = log_loss(y_test_float, test_proba)
cm = confusion_matrix(y_test_float, test_preds)

print("\n=== RESULTADOS EN TEST ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1 macro:  {f1_macro:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"Log Loss:  {ll:.4f}")

print("\nMatriz de confusión:")
print(cm)


=== RESULTADOS EN TEST ===
Accuracy:  0.5703
F1 macro:  0.5703
ROC-AUC:   0.6002
Log Loss:  0.6768

Matriz de confusión:
[[197756 154611]
 [148202 204165]]


### 5.3 Evaluación de Modelos con Team Differential